# 06 · Completion and swaps (spec §7.4)

Beam search over legal partial decks (legality only from `crdeck.rules`). Objective, with weights from config:

`meta_win_prob + synergy_weight · mean_pair_synergy + cooccurrence_weight · mean_log_lift`

The co-occurrence term (do players actually pair these cards?) is tuned for completion recall on the **validation** window only.

In [ ]:
import os
import json
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

DATA_SOURCE = os.environ.get("CRDECK_DATA_SOURCE", "real")  # "real" (default) or "synthetic"
from crdeck.config import Paths
paths = Paths.for_source(DATA_SOURCE)
print(paths.banner())

In [ ]:
from crdeck.pipeline import load_context
ctx = load_context(DATA_SOURCE)  # raises NoDataError with instructions if data is missing; never substitutes
print(ctx.paths.banner())
ctx.split.describe()

In [ ]:
from crdeck.evaluate import tune_cooccurrence, choose_cooccurrence
tuning = tune_cooccurrence(ctx)
chosen = choose_cooccurrence(tuning)
print("config cooccurrence_weight:", ctx.cfg["search"]["cooccurrence_weight"], "| chosen on validation:", chosen)
tuning

If the chosen value differs from `config/model.yaml`, update the config (it is a starting value tuned on synthetic data).

## Completion recall@10 on held-out decks (test window)
Mask 2 cards from frequent test-window decks; does the model's ranked fill contain them? Baseline: most popular cards.

In [ ]:
from crdeck.evaluate import completion_recall
w = chosen if chosen is not None else ctx.cfg["search"]["cooccurrence_weight"]
completion_recall(ctx, n_decks=100, cooccurrence_weight=w)

## Swaps
For each non-seed card, the top legal replacements with the objective change and the pair evidence.

In [ ]:
from crdeck.search import Searcher, parse_seed
s = Searcher(ctx, w)
seed = [parse_seed("Golem", ctx.cards)]
deck = s.complete(seed)[0]
print([s._name(e) for e in deck.entries])
pd.DataFrame(s.swaps(deck.entries, locked=seed)).drop(columns=["new_deck_key"]).head(12)